# JEBAT-BUILDER Fine-Tuning

Fine-tune qwen2.5-coder:7b on JEBAT codebase using Unsloth + QLoRA.

**Setup:** Runtime > Change runtime type > T4 GPU

**Steps:**
1. Upload `training_data.jsonl` to Colab Files panel
2. Run all cells in order
3. Download the GGUF file when done
4. Deploy to .206 server

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth
!pip install -q --upgrade transformers peft accelerate trl
!pip install -q torch
print("Dependencies installed")

In [ ]:
# Cell 2: Load model with LoRA
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA rank: {LORA_RANK}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 3: Load training data
def load_training_data(file_path):
    examples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            instruction = data['instruction']
            input_ctx = data.get('input', '')
            output = data['output']
            if input_ctx:
                prompt = f"{instruction}\n\nContext: {input_ctx}"
            else:
                prompt = instruction
            text = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
            examples.append({"text": text})
    return Dataset.from_list(examples)

dataset = load_training_data("training_data.jsonl")
print(f"Loaded {len(dataset)} training examples")

# Show a sample
print("\n--- Sample ---")
print(dataset[0]["text"][:500])
print("...")

In [ ]:
# Cell 4: Configure and run training
OUTPUT_DIR = "./jebat-builder-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="epoch",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=True,
)

print("Starting fine-tuning...")
print(f"Epochs: 3 | Batch: 2 | Accumulation: 4 | Effective batch: 8")
print(f"Learning rate: 2e-4 | LoRA rank: 16")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 5: Save and export to GGUF
import os

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# Export to GGUF
print("Exporting to GGUF (q4_k_m)...")
model.save_pretrained_gguf("jebat-builder", tokenizer, quantization_method="q4_k_m")
print("GGUF export complete")

# List files
for root, dirs, files in os.walk("jebat-builder"):
    for f in files:
        if f.endswith(".gguf"):
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024 / 1024
            print(f"\nGGUF file: {path}")
            print(f"Size: {size:.1f} MB")

In [ ]:
# Cell 6: Test the fine-tuned model
FastLanguageModel.for_inference(model)

test_prompts = [
    "Create a FastAPI endpoint with JWT authentication and rate limiting",
    "Write a Python port scanner with multi-threading and banner grabbing",
    "Design a multi-tenant PostgreSQL schema with RLS",
    "How do I use JEBAT multi-agent system to build a full-stack app?",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}")
    inputs = tokenizer([f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, use_cache=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "<|im_start|>assistant\n" in response:
        response = response.split("<|im_start|>assistant\n")[-1]
    print(response[:500])
    print("...")

In [ ]:
# Cell 7: Download instructions
import glob

gguf_files = glob.glob("jebat-builder/*.gguf")
if gguf_files:
    gguf_path = gguf_files[0]
    print(f"GGUF ready: {gguf_path}")
    print()
    print("To deploy to .206 server:")
    print(f"  1. Download {gguf_path} from Colab Files panel")
    print(f"  2. Upload to server:")
    print(f"     scp {gguf_path} root@72.62.255.206:/tmp/jebat-builder.gguf")
    print(f"  3. Create Ollama model:")
    print(f"     ssh root@72.62.255.206 'ollama create jebat-builder -f /tmp/Modelfile' ")
    print(f"  4. Test:")
    print(f"     curl https://jebat.online/ollama/api/tags")
    print()
    print("To use locally with llama.cpp:")
    print(f"  llama-server -m {gguf_path} --port 8080")
else:
    print("No GGUF files found. Check export step.")